In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from matplotlib.ticker import MaxNLocator
import matplotlib as mpl
new_rc_params = {'text.usetex': False, "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)

In [ ]:
#read stiffness measurements
df=pd.read_excel('/g/korbel/olisov/Visium_Linda/stiffness/stiffness_measurements_50um/merged/260119_errorCorrected_mappedDF_filtered_forDima.xlsx', index_col=0)

In [ ]:
#add section(sample) to the barcode
df['Barcode']=df['section']+'_'+df['Barcode']

In [ ]:
# add spot pathologist annotation
def load_and_annotate_csv_files(folder_path):
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    annotated_dfs = []
    
    for csv_file in csv_files:
        df = pd.read_csv(os.path.join(folder_path, csv_file), names=['Barcode', 'Annotation'], skiprows=1)
        prefix = csv_file.split('_')[0]
        df['Barcode'] = df['Barcode'].apply(lambda x: f"{prefix}_{x}")
        annotated_dfs.append(df)
    
    concatenated_df = pd.concat(annotated_dfs, ignore_index=True)
    return concatenated_df

# Path to the folder containing the CSV files
folder_path = '/g/korbel/olisov/Visium_Linda/Spot_Annotation'

# Load and annotate the CSV files
annotated_df = load_and_annotate_csv_files(folder_path)

In [ ]:
# Merge with combined_df on the 'Barcode' column
combined_df = pd.merge(df, annotated_df, on='Barcode', how='left')

#keep 'Stroma' annotation
combined_df=combined_df[combined_df['Annotation']=='Stroma']
combined_df = combined_df.reset_index(drop=True)

In [ ]:
#plot  measurements per spot
# Count the occurrences of each unique barcode
barcode_counts = combined_df['Barcode'].value_counts()

# Calculate the median of the counts
median_count = barcode_counts.median()

# Plot the histogram
plt.figure(figsize=(10, 6))
plt.hist(barcode_counts, bins=range(1, barcode_counts.max() + 2), alpha=0.7, color='blue', edgecolor='black')
plt.axvline(median_count, color='red', linestyle='dashed', linewidth=1, label=f'Median')
plt.title('Counts of measurements across capture spots')
plt.xlabel('Counts of Measurements')
plt.ylabel('Frequency')
plt.legend(frameon=False)
plt.savefig('/g/korbel/olisov/Visium_Linda/paper_figures/11samples/QC_supplements/measurements_hist_11samples.svg')
plt.show()

In [ ]:
# Calculate the counts of each unique value in the 'Barcode' column
barcode_counts = combined_df['Barcode'].value_counts()

# Identify barcodes that appear less than two times
barcodes_to_keep = barcode_counts[barcode_counts >= 2].index

# Keep rows where the 'Barcode' value appears two or more times
df = combined_df[combined_df['Barcode'].isin(barcodes_to_keep)]

# Filtering of measurements based on MAD and CV 

In [ ]:
from sklearn.preprocessing import RobustScaler

# Get the threshold
mad_threshold = 3.5
cv_threshold = 40 

# Function to compute Median Absolute Deviation (MAD) based outliers
def apply_mad(dataframe, column, threshold=mad_threshold):
    # Scale data using RobustScaler
    scaler = RobustScaler()
    reshaped_data = dataframe[column].values.reshape(-1, 1)
    scaled_data = scaler.fit_transform(reshaped_data)

    # Calculate MAD
    median = np.median(scaled_data)
    diff = np.abs(scaled_data - median)
    mad = np.median(diff)
    
    # Define outliers as those where the deviation is more than 'threshold' times the MAD
    modified_z_score = 0.6745 * diff / mad
    return dataframe[modified_z_score > threshold]

# Function to compute Coefficient of Variation (CV) based filtering for two measurements
def apply_cv_filter(dataframe, column, cv_threshold):
    mean_value = np.mean(dataframe[column])
    std_dev = np.std(dataframe[column])
    cv = std_dev * 100 / mean_value if mean_value != 0 else 0
    if cv > cv_threshold:
        # If the CV is too high, consider both points as outliers
        return dataframe  # Return both measurements
    else:
        return pd.DataFrame()  # No outliers

unique_barcodes = df['Barcode'].unique()
# Collect non-outlier data
filtered_data = pd.DataFrame()

for barcode in unique_barcodes:
    data = df[df['Barcode'] == barcode]
    # Remove outliers using MAD value if more than 2 measurements
    if len(data) > 2:  
        outliers_df = apply_mad(data, 'Y', threshold=mad_threshold)
        data = data[~data.index.isin(outliers_df.index)]
        # Append non-outlier data to filtered_data
        filtered_data = pd.concat([filtered_data, data])
    elif len(data) == 2:  # If there are exactly 2 measurements, apply CV-based filter
        outliers_df = apply_cv_filter(data, 'Y', cv_threshold=cv_threshold)
        data = data[~data.index.isin(outliers_df.index)]
        filtered_data = pd.concat([filtered_data, data])

In [ ]:
filtered_data.to_csv('/g/korbel/olisov/Visium_Linda/stiffness/stiffness_measurements_50um/merged/11_samples_mapping_final.tsv', sep='\t', index=False)